# NB15 — NETL Produced Waters Replication

Tests the primary per-Mb specialization finding in **NETL produced waters** —
hydraulically fractured well samples with extremely high metal concentrations.
These samples are a strong positive-control environment for metal resistance.

**Dataset**: `netl_pw_dna` Spark namespace — 16S V4 amplicon (primers 515F/806R)
from produced water and associated reference environments.

**Niche metric**: Shannon entropy across well types / sample environment categories.
Wider distribution across environment types = higher H = more cosmopolitan.

**Status**: ⏳ Pending — schema exploration first, then compute niche metric.


## Block 0 — Spark setup

In [ ]:
try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    spark = SparkSession.builder.getOrCreate()
    print(f'Spark version: {spark.version}')
    SPARK = True
except Exception as e:
    print(f'Spark not available: {e}')
    print('Run this notebook on JupyterHub for Spark access.')
    SPARK = False

In [ ]:
if SPARK:
    tables = spark.sql('SHOW TABLES IN netl_pw_dna').toPandas()
    print('Tables in netl_pw_dna:')
    print(tables.to_string())
else:
    print('Skipping — no Spark')

## Block 1 — Schema exploration

In [ ]:
if SPARK:
    for tbl in ['amplicon_515f_806r', 'amplicon_27f_519', 'qiime_counts',
                'study_sample_metadata', 'feature_annotations']:
        try:
            print(f'\n=== {tbl} ===')
            desc = spark.sql(f'DESCRIBE netl_pw_dna.{tbl}').toPandas()
            print(desc.to_string())
            cnt = spark.sql(f'SELECT COUNT(*) as n FROM netl_pw_dna.{tbl}').collect()[0].n
            print(f'Rows: {cnt:,}')
        except Exception as e:
            print(f'  ERROR: {e}')

In [ ]:
if SPARK:
    for tbl in ['amplicon_515f_806r', 'study_sample_metadata', 'feature_annotations']:
        try:
            print(f'\n=== {tbl} (first 3 rows) ===')
            df = spark.sql(f'SELECT * FROM netl_pw_dna.{tbl} LIMIT 3').toPandas()
            print(df.to_string())
        except Exception as e:
            print(f'  ERROR: {e}')

## Block 2 — Taxonomy and genus mapping

In [ ]:
if SPARK:
    # Check taxonomy format in feature_annotations
    print('=== feature_annotations taxonomy sample ===')
    try:
        tax_df = spark.sql("""
            SELECT *
            FROM netl_pw_dna.feature_annotations
            LIMIT 20
        """).toPandas()
        print(tax_df.to_string())
        print()
        # Check for genus-level taxonomy columns
        print('Columns:', tax_df.columns.tolist())
    except Exception as e:
        print(f'ERROR: {e}')

In [ ]:
if SPARK:
    # Try to extract genus from taxonomy string (SILVA format: d__Bacteria;p__...;g__Genus;...)
    try:
        tax_cols = spark.sql('DESCRIBE netl_pw_dna.feature_annotations').toPandas()
        print('feature_annotations columns:', tax_cols['col_name'].tolist())
    except Exception as e:
        print(f'ERROR: {e}')
    print()
    print('Decision gate: if taxonomy is SILVA-based (not GTDB), check genus name overlap')
    print('with primary analysis before proceeding to PGLS.')

## Block 3 — Sample metadata and environment categories

In [ ]:
if SPARK:
    try:
        meta = spark.sql("""
            SELECT *
            FROM netl_pw_dna.study_sample_metadata
            LIMIT 20
        """).toPandas()
        print('study_sample_metadata (20 rows):')
        print(meta.to_string())
        print()
        print('Unique values in potentially useful columns:')
        for col in meta.columns:
            n_unique = meta[col].nunique()
            if 2 <= n_unique <= 30:
                print(f'  {col}: {sorted(meta[col].dropna().unique().tolist())}')
    except Exception as e:
        print(f'ERROR: {e}')

In [ ]:
if SPARK:
    # Get all unique environment/sample types for niche metric design
    try:
        env_counts = spark.sql("""
            SELECT *
            FROM netl_pw_dna.study_sample_metadata
        """).toPandas()
        print(f'Total samples: {len(env_counts):,}')
        # Print distribution of any categorical column
        for col in env_counts.columns:
            n_unique = env_counts[col].nunique()
            if 2 <= n_unique <= 20 and env_counts[col].dtype == object:
                print(f'\n{col} ({n_unique} categories):')
                print(env_counts[col].value_counts())
    except Exception as e:
        print(f'ERROR: {e}')

## Block 4 — Amplicon data extraction

*(Complete after Block 1-3 inspection reveals table structure)*

In [ ]:
# Template: adapt column names based on Block 1-3 output
# Requires: feature ID column, sample ID column, count column in amplicon table
# Requires: taxonomy column in feature_annotations (genus extraction)
# Requires: environment category column in study_sample_metadata

if SPARK:
    # Step 1: join amplicon counts × taxonomy × metadata
    # (adapt column names after schema inspection)
    print('TODO: implement after schema inspection in Block 1-3')
    print('Expected output: genus × sample_type count matrix')
    print('Then: compute Shannon entropy per genus, merge with 94-KO/Mb, run PGLS')